# syntax

> The spaCy rules: sentences, clauses, passives, and the patterns anchored to them

In [1]:
#| default_exp syntax

In [2]:
#| hide
from nbdev.showdoc import *

The rules that need to know what a sentence is and how its words relate: sentence length, clause count, passive voice, part-of-speech-dependent vocabulary, noun clusters, nominalizations, agent defocusing, and the slop patterns that anchor to sentence starts. [spaCy](https://spacy.io) provides the linguistic analysis, and this module also owns getting spaCy's model onto the machine.

In [3]:
#| export
import spacy, zipfile, shutil, importlib
from fastcore.utils import *
from fastcore.net import urlsave
from fastcore.xdg import xdg_cache_home
from slopometer.core import *
from slopometer.segment import *

In [4]:
from fastcore.test import *

## Getting the model

slopometer uses spaCy's `en_core_web_md` pipeline. The lg and md English models share one vocabulary of 684,830 keys and the same tagger and parser architecture. They differ in the vector table: md maps the vocabulary onto 20,000 shared vectors where lg keeps 342,918. That difference mattered when the design called for vector-similarity rules. The para notebook then measured those designs and they failed on real fixtures, for reasons unrelated to vector quality. What survives needs only the vocabulary key set, which the coinage rule reads through `is_oov`, and md and lg answer that identically. So the small model won: 40MB against 560MB, with nothing shipped that the difference could affect. A future rule that needs calibrated similarity brings the lg question back, and this paragraph is where to reopen it.

The model is a wheel that Explosion publishes on [GitHub releases](https://github.com/explosion/spacy-models/releases), not on PyPI, and published packages cannot carry direct-URL dependencies. So slopometer follows the pattern that huggingface_hub, tiktoken, and playwright settled on for artifacts too big to package: download once into a user cache directory, and load by path. `pip install slopometer` is complete by itself. The first call on a machine without the model downloads it, says so, and never needs to again. The cache lives outside every virtual environment, one copy serves every project, and environment syncs cannot remove it.


In [5]:
#| export
MODEL = 'en_core_web_md-3.8.0'
MODEL_PKG = MODEL.rsplit('-', 1)[0]
MODEL_URL = f'https://github.com/explosion/spacy-models/releases/download/{MODEL}/{MODEL}-py3-none-any.whl'

def model_path():
    "Path of the model data directory: the installed package if present, else the cache, downloading on first call"
    try: return Path(importlib.import_module(MODEL_PKG).__file__).parent/MODEL
    except ImportError: pass
    dest = xdg_cache_home()/'slopometer'
    p = dest/MODEL
    if p.exists(): return p
    print(f'slopometer: downloading {MODEL} (~40MB, once per machine) to {p}')
    dest.mkdir(parents=True, exist_ok=True)
    whl = urlsave(MODEL_URL, dest/f'{MODEL}.whl')
    with zipfile.ZipFile(whl) as z:
        pfx = f'{MODEL_PKG}/{MODEL}/'
        for n in z.namelist():
            if n.startswith(pfx): z.extract(n, dest/'tmp')
    shutil.move(dest/'tmp'/MODEL_PKG/MODEL, p)
    shutil.rmtree(dest/'tmp')
    Path(whl).unlink()
    return p

_nlp = None
def get_nlp():
    "The loaded pipeline, one per process"
    global _nlp
    if _nlp is None: _nlp = spacy.load(model_path())
    return _nlp

In [6]:
p = model_path()
p.exists(), p.name

(True, 'en_core_web_md-3.8.0')

The download branch runs only on a machine with neither the package nor the cache, and exercising it costs a 560MB download, so no test pins it. CI restores the cache directory with `actions/cache` instead. `get_nlp` loads once per process: the load takes seconds, every later call is free, and the CLI notebook builds on exactly that asymmetry.

## Reading a parse

Everything below uses three layers of spaCy's analysis, and they are worth meeting before any rule uses them. A token is one word or punctuation mark. Its part of speech (`pos_`) says what kind of word it is: `NOUN`, `VERB`, `ADJ`, `ADV`. Its lemma (`lemma_`) is the dictionary form, which is how one rule entry catches "leverages" and "leveraged". The dependency parse (`dep_`, `head`) links every token to the token it modifies, labelled with the grammatical relation: `nsubj` marks a verb's subject, `dobj` its object. Sentence boundaries fall out of the same parse, and every sentence-level rule trusts them rather than splitting on periods, which fail on "e.g." and "v3.8.0".

spaCy's own [spaCy 101](https://spacy.io/usage/spacy-101) is the practical tour. The label scheme descends from [Universal Dependencies](https://universaldependencies.org/), and chapters 17-18 of Jurafsky and Martin's [Speech and Language Processing](https://web.stanford.edu/~jurafsky/slp3/) explain how statistical parsers assign these structures, and why they are occasionally wrong. The rules below treat the parse as evidence, not truth, and each one's thresholds are set so that a mildly wrong parse produces a missed finding rather than a false one.

In [7]:
doc = get_nlp()('The gateway never kills an unresponsive kernel.')
L((t.text, t.pos_, t.lemma_, t.dep_, t.head.text) for t in doc)

[('The', 'DET', 'the', 'det', 'gateway'), ('gateway', 'NOUN', 'gateway', 'nsubj', 'kills'), ('never', 'ADV', 'never', 'neg', 'kills'), ('kills', 'VERB', 'kill', 'ROOT', 'kills'), ('an', 'DET', 'an', 'det', 'kernel'), ('unresponsive', 'ADJ', 'unresponsive', 'amod', 'kernel'), ('kernel', 'NOUN', 'kernel', 'dobj', 'kills'), ('.', 'PUNCT', '.', 'punct', 'kills')]

## Sentence load

ASD-STE100 caps sentences at 20 words for procedures and 25 for descriptions, on the evidence that comprehension drops steeply past that. Reference prose is descriptive, and the cap here is 25, counting words rather than punctuation. The weight escalates by one per word over the cap, which makes a 40-word sentence an alarm rather than a nudge. Clause count is the same idea measured structurally, and the counting follows what a reader must track rather than raw verb tallies. A subordinate clause (`advcl`, `ccomp`) is an idea. A coordinated verb counts only when it brings its own subject: "the kernel gains ports and clients see restarting" stacks two ideas, where "reads the same stdin, writes to the same terminal, sees the same directory" enumerates one. A relative clause completes its noun ("a small program that starts fast") and does not count at all. Three ideas is ordinary English. The fourth begins the pile-up, and the weight escalates from there.

In [8]:
#| export
@rule('sent_len', tell=1, weight=PRESSURE, level='sentence')
def find_sent_len(sent):
    "Sentences past 25 words, escalating one weight point per extra word"
    n = sum(1 for t in sent if not t.is_punct)
    if n <= 25: return []
    return [Finding('sent_len', 1, 0, len(sent.text), sent.text, PRESSURE*(n-25))]

_clausy = {'advcl', 'ccomp'}

def _own_subj(t): return any(c.dep_ in ('nsubj', 'nsubjpass', 'expl') for c in t.children)

@rule('clauses', tell=1, weight=PRESSURE, level='sentence')
def find_clauses(sent):
    "Sentences stacking four or more idea-bearing clauses, escalating per extra clause"
    n = 1 + sum(1 for t in sent if t.pos_ in ('VERB', 'AUX')
        and (t.dep_ in _clausy or (t.dep_ == 'conj' and _own_subj(t))))
    if n < 4: return []
    return [Finding('clauses', 1, 0, len(sent.text), sent.text, PRESSURE*(n-3))]

In [9]:
long_sent = list(get_nlp()("It terminates and respawns; the kernel gains fresh ports, fresh channels, and a fresh interpreter via the new process, so the channel set is rebuilt and clients simply see restarting then a fresh welcome-backed ready kernel.").sents)[0]
enum = list(get_nlp()('Your function gets the same arguments, reads the same stdin, writes to the same terminal, and produces the same exit code.').sents)[0]
test_eq(find_clauses(enum), [])
find_sent_len(long_sent) + find_clauses(long_sent)

[[12] sent_len (tell 1, splices): 'It terminates and respawns; the kernel gains fresh ports, fresh channels, and a fresh interpreter via the new process, so the channel set is rebuilt and clients simply see restarting then a fresh welcome-backed ready kernel.',
 [1] clauses (tell 1, splices): 'It terminates and respawns; the kernel gains fresh ports, fresh channels, and a fresh interpreter via the new process, so the channel set is rebuilt and clients simply see restarting then a fresh welcome-backed ready kernel.']

## Passive voice

`write_docs` prefers active voice everywhere, and its reason is sharper than style: a passive that hides the actor usually hides part of the contract with it. "The file is deleted" leaves the deleter unnamed, and the deleter is what the reader needed. The parse makes the distinction mechanical. A passive subject carries the `nsubjpass` label, and when the verb also has an `agent` child, the actor survives in a "by" phrase. An agentless passive weighs more than an agented one, because the agented form at least kept the contract on the page.

The label names are an external assumption worth pinning: spaCy's English models use the ClearNLP label set, not bare Universal Dependencies, and a model change that renamed `nsubjpass` would silently blind this rule. The assertion below is the tripwire.

In [10]:
#| export
@rule('passive', tell=None, weight=SMELL, level='sentence')
def find_passive(sent):
    "Passive constructions, weighted higher when no agent survives"
    res = []
    for t in sent:
        if t.dep_ != 'nsubjpass': continue
        v = t.head
        agented = any(c.dep_ == 'agent' for c in v.children)
        span = sent.text[t.idx-sent.start_char:v.idx-sent.start_char+len(v.text)]
        res.append(Finding('passive', None, t.idx-sent.start_char, v.idx-sent.start_char+len(v.text), span, PRESSURE if agented else SMELL))
    return res

In [11]:
ps = list(get_nlp()('The config file is read at startup. The socket was closed by the client.').sents)
test_eq([t.dep_ for t in ps[0]][:4], ['det', 'compound', 'nsubjpass', 'auxpass'])
find_passive(ps[0]) + find_passive(ps[1])

[[3] passive: 'file is read', [1] passive: 'socket was closed']

## Vocabulary judged by the parse

The lexicon notebook flags words that are wrong in every part of speech. This rule covers the rest: words that are banned only as verbs. "The impact was small" states a measurement, and "this impacts performance" inflates it. "The fix landed" is the metaphor `write_docs` calls out by name, and its replacement names what actually happened: merged, committed, released. The parse's POS tag is the disambiguator, and the lemma catches every inflection. A token preceded by a hyphen is part of a compound like "aviation-shaped" and never flags.

The intensifier rule is tell 3's adverb half. "simply" and "merely" claim an ease the reader cannot verify, and deleting them never changes a contract. "just" stays out on precision grounds: its minimizing sense ("it just works") and its temporal sense ("just released") wear the same POS tag, and flagging releases notes for recency would teach readers to ignore the rule.

In [12]:
#| export
_verb_banned = dict(impact='affect', land=None, shape=None, ride=None)
_intens = {'simply', 'merely', 'effortlessly'}

def _tok_find(name, tell, weight, sent, pred, lex=None):
    "Findings for tokens of `sent` passing `pred`, with optional suggestions from `lex`"
    return [Finding(name, tell, t.idx-sent.start_char, t.idx-sent.start_char+len(t.text), t.text, weight,
        lex.get(t.lemma_) if lex else None) for t in sent if pred(t)]

@rule('verb_banned', tell=None, weight=KILL, level='sentence')
def find_verb_banned(sent):
    "Words banned as verbs, caught by lemma and POS"
    return _tok_find('verb_banned', None, KILL, sent, lambda t: t.pos_ == 'VERB' and t.lemma_ in _verb_banned
        and not (t.i > 0 and t.nbor(-1).text == '-'), _verb_banned)

@rule('intensifiers', tell=3, weight=SMELL, level='sentence')
def find_intensifiers(sent):
    "Ease-claiming adverbs"
    return _tok_find('intensifiers', 3, SMELL, sent, lambda t: t.pos_ == 'ADV' and t.lemma_ in _intens)

In [13]:
s1 = list(get_nlp()('The fix landed in main, and clients simply see the new behavior.').sents)[0]
test_eq(find_verb_banned(list(get_nlp()('The impact was small.').sents)[0]), [])
test_eq(find_verb_banned(list(get_nlp()('These rules are aviation-shaped.').sents)[0]), [])
find_verb_banned(s1) + find_intensifiers(s1)

[[10] verb_banned: 'landed',
 [3] intensifiers (tell 3, emphasis devices): 'simply']

## Noun clusters and nominalizations

ASD-STE100 limits noun clusters to three, because "kernel connection state model update" makes the reader do the parser's job: which noun modifies which is unstated, and every reader guesses alone. The rule counts only common nouns toward its threshold of three, so "New York City subway" passes as a name with one noun while "kernel connection state model" flags as a construction. The weight escalates per extra noun.

A nominalization buries a verb inside a noun and then needs a light verb to carry the sentence: "make a decision" for "decide", "perform an assessment" for "assess". The pattern is a light verb with a derived-noun object, and the suggestion is the buried verb where the mapping is mechanical.

In [14]:
#| export
_light = {'make', 'take', 'perform', 'conduct', 'do', 'carry'}
_denoun = dict(decision='decide', assessment='assess', improvement='improve', comparison='compare',
    configuration='configure', modification='modify', evaluation='evaluate', determination='determine',
    implementation='implement', investigation='investigate', selection='select', validation='validate')

@rule('noun_cluster', tell=None, weight=PRESSURE, level='sentence')
def find_noun_cluster(sent):
    "Runs of three or more nouns, escalating per extra noun"
    res,run = [],[]
    for t in list(sent) + [None]:
        if t is not None and t.pos_ in ('NOUN', 'PROPN'):
            run.append(t)
            continue
        n = sum(x.pos_ == 'NOUN' for x in run)
        if n >= 3:
            span = sent.text[run[0].idx-sent.start_char:run[-1].idx-sent.start_char+len(run[-1].text)]
            res.append(Finding('noun_cluster', None, run[0].idx-sent.start_char, run[-1].idx-sent.start_char+len(run[-1].text), span, PRESSURE*(n-2)))
        run = []
    return res

@rule('nominalization', tell=None, weight=SMELL, level='sentence')
def find_nominalization(sent):
    "Light verbs carrying a nominalized object"
    res = []
    for t in sent:
        if t.pos_ == 'VERB' and t.lemma_ in _light:
            for c in t.children:
                if c.dep_ == 'dobj' and c.lemma_ in _denoun:
                    span = sent.text[t.idx-sent.start_char:c.idx-sent.start_char+len(c.text)]
                    res.append(Finding('nominalization', None, t.idx-sent.start_char, c.idx-sent.start_char+len(c.text), span, SMELL, _denoun[c.lemma_]))
    return res

In [15]:
s2 = list(get_nlp()('The kernel connection state model update takes a decision about restarts.').sents)[0]
test_eq(find_noun_cluster(list(get_nlp()('The New York City subway runs all night.').sents)[0]), [])
find_noun_cluster(s2) + find_nominalization(s2)

[[2] noun_cluster: 'kernel connection state model update',
 [3] nominalization: 'takes a decision' -> 'decide']

## Agent defocusing

Tells 22 and 23 are two faces of one grammatical device, which Joseph Williams's *Style: Toward Clarity and Grace* treats as the central sin of institutional prose: pushing the true actor out of the subject seat. In artifact-as-agent (tell 22), an authored thing takes credit for its author's deed: "this PR introduces". In recipient-as-subject (tell 23), the beneficiary takes the seat and the doer hides in a "via" phrase: "the kernel gains fresh ports via the new process". The passive is the third face, and its rule is above. Williams's positive principle names the fix: make the doer the subject and the deed the verb.

Both rules trade recall for precision. Artifact-as-agent fires only when the subject is an authorship artifact and the verb is a crediting verb, because "the parser rejects malformed input" is a tool doing exactly what it does and must pass. Recipient-as-subject fires on "gain" and "receive" outright, and on "get" only when a "via" or "through" phrase marks the demoted doer.

In [16]:
#| export
_artifacts = {'pr', 'commit', 'change', 'patch', 'release', 'update', 'design', 'proposal', 'document', 'doc', 'readme', 'section', 'paragraph', 'refactor'}
_credit_verbs = {'introduce', 'enable', 'allow', 'let', 'add', 'improve', 'fix', 'ensure', 'provide', 'bring', 'unlock'}

@rule('artifact_agent', tell=22, weight=SMELL, level='sentence')
def find_artifact_agent(sent):
    "Authored artifacts credited with their author's deed"
    res = []
    for t in sent:
        if t.dep_ == 'nsubj' and t.lemma_.lower() in _artifacts and t.head.lemma_ in _credit_verbs:
            span = sent.text[t.idx-sent.start_char:t.head.idx-sent.start_char+len(t.head.text)]
            res.append(Finding('artifact_agent', 22, t.idx-sent.start_char, t.head.idx-sent.start_char+len(t.head.text), span, SMELL))
    return res

@rule('recipient_subject', tell=23, weight=SMELL, level='sentence')
def find_recipient_subject(sent):
    "Beneficiaries promoted to subject with gain/receive, or get with a via phrase"
    res = []
    for t in sent:
        if t.pos_ != 'VERB' or t.lemma_ not in ('gain', 'receive', 'get'): continue
        if not any(c.dep_ == 'nsubj' for c in t.children): continue
        demoted = any(c.lemma_ in ('via', 'through') for c in t.subtree if c.pos_ == 'ADP')
        if t.lemma_ == 'get' and not demoted: continue
        res.append(Finding('recipient_subject', 23, t.idx-sent.start_char, t.idx-sent.start_char+len(t.text), t.text, SMELL))
    return res

In [17]:
s3 = list(get_nlp()('This PR introduces retry logic, and the kernel gains fresh ports via the new process.').sents)[0]
test_eq(find_artifact_agent(list(get_nlp()('The parser rejects malformed input.').sents)[0]), [])
find_artifact_agent(s3) + find_recipient_subject(s3)

[[3] recipient_subject (tell 23, recipient-as-subject): 'gains']

## Sentence-anchored patterns

Seven tells are fixed phrases that live at known positions in a sentence, and sentence boundaries are what the parse contributes: "This section describes" is throat-clearing at a sentence start and ordinary prose in the middle of one ("the part this section describes"). Each rule is a handful of curated phrases with near-zero false-positive rates, and each list is deliberately short: a phrase joins when it is slop wherever it appears in reference prose, and not before. Rhetorical questions (tell 18) need no phrase list at all, since reference prose never asks the reader anything: a sentence ending in a question mark is the finding. Not-X-but-Y (tell 16) is the one kill-on-sight pattern, and its list sticks to the unambiguous "not just/merely/only" skeletons rather than every "not X, but Y", because "not thread-safe, but lockable" is a contract stating both halves.

In [18]:
#| export
def sent_rule(
    name, # Rule name, as in `core.rule`
    tell, # `write_docs` tell number
    weight, # Tier the findings carry
    pats, # Phrase list; each becomes a case-insensitive regex alternative
    anchored=False, # Match only at the sentence start?
):
    "Build and register a sentence rule that scans `pats` in the sentence text"
    pat = re.compile(('^' if anchored else '') + '(' + '|'.join(pats) + ')', re.I)
    def _f(sent): return [Finding(name, tell, m.start(), m.end(), m.group(), weight) for m in pat.finditer(sent.text)]
    _f.__name__,_f.__doc__ = f'find_{name}',f'Scan for {name} phrases'
    return rule(name, tell=tell, weight=weight, level='sentence')(_f)

find_throat = sent_rule('throat_clearing', 13, SMELL, [r"this (?:section|document|readme|page|guide|module|doc|notebook) (?:describes|covers|explains|documents|provides|outlines|introduces)", r'the purpose of this'], anchored=True)
find_todays = sent_rule('todays_world', 14, SMELL, [r"in today's"], anchored=True)
find_announce = sent_rule('announce', 15, SMELL, [r"(?:the (?:fix|answer|solution|key|result|takeaway|core \w+)|here's the thing|bottom line): "], anchored=True)
find_notxbuty = sent_rule('notxbuty', 16, KILL, [r"\bis(?:n't| not) (?:just|merely|simply|only) (?:a|an|the|about)\b", r"\baren't (?:just|merely|simply|only)\b", r"\bit's not about\b", r"\bnot (?:just|merely) about\b"])
find_teaser = sent_rule('teaser', 17, SMELL, [r"here's where it gets", r'\bthe real story\b', r'\bgets (?:really )?interesting\b', r'\bthe main event\b'])
find_appraisal = sent_rule('appraisal', 21, SMELL, [r'\bworth being precise\b', r"\bthe key (?:point|insight|thing) is\b", r"\bwhat's interesting is\b", r'\bcrucially\b', r'\bthe important thing is\b'])

@rule('rhetorical', tell=18, weight=SMELL, level='sentence')
def find_rhetorical(sent):
    "Sentences that ask the reader a question"
    t = sent.text.rstrip()
    if not t.endswith('?'): return []
    return [Finding('rhetorical', 18, 0, len(t), t, SMELL)]

In [19]:
checks = ["This section describes how the gateway manages the kernel lifecycle.",
    "So what does restart actually do?", "It isn't just a poller - it's the liveness authority.",
    "The distinction is worth being precise about.", "The core mechanism: watch."]
found = L(f for s in checks for sent in get_nlp()(s).sents
    for f in find_throat(sent)+find_rhetorical(sent)+find_notxbuty(sent)+find_appraisal(sent)+find_announce(sent))
test_eq(find_notxbuty(list(get_nlp()('The list is not just names, and callers must not assume order.').sents)[0]), [])
found

[[3] throat_clearing (tell 13, throat-clearing): 'This section describes', [3] rhetorical (tell 18, rhetorical questions): 'So what does restart actually do?', [10] notxbuty (tell 16, not-X-but-Y): "isn't just a", [3] appraisal (tell 21, appraisal preamble): 'worth being precise', [3] announce (tell 15, announce-then-deliver): 'The core mechanism: ']

In [ ]:
#| hide
import nbdev
nbdev.nbdev_export()